In [ ]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer


###############################################################
# SAMPLE DATA GENERATOR
###############################################################

def create_sample_hr_dataset(file_path):

    sample_data = [
        "Employees are entitled to 25 days of annual leave per year.",
        "Sick leave must be reported to the manager before 9 AM.",
        "Remote work is allowed up to 3 days per week depending on role.",
        "Employees must complete mandatory compliance training annually.",
        "Parental leave is available according to statutory regulations.",
        "Employees should follow the company code of conduct at all times.",
        "Expense claims must be submitted within 30 days of purchase.",
        "Flexible working requests can be submitted after 6 months of employment.",
        "Employees are entitled to workplace safety training.",
        "IT equipment must be returned upon termination of employment."
    ]

    df = pd.DataFrame(sample_data, columns=["text"])
    df.to_csv(file_path, index=False)

    print("Sample HR dataset created.")


###############################################################
# SEMANTIC SEARCH ENGINE
###############################################################

class SemanticSearchEngine:

    def __init__(self, data_path):

        if not os.path.exists(data_path):
            print("Dataset missing. Creating sample dataset...")
            create_sample_hr_dataset(data_path)

        self.df = pd.read_csv(data_path)

        if "text" not in self.df.columns:
            raise ValueError("Dataset must contain a 'text' column")

        print("Loading embedding model...")
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

        print("Computing embeddings...")
        self.embeddings = self.model.encode(
            self.df["text"].tolist(),
            convert_to_numpy=True
        )

        print("Embeddings ready.\n")


    def search(self, query, top_k=3):

        query_embedding = self.model.encode(
            query,
            convert_to_numpy=True
        )

        similarities = np.dot(
            self.embeddings,
            query_embedding
        ) / (
            np.linalg.norm(self.embeddings, axis=1)
            * np.linalg.norm(query_embedding)
        )

        top_indices = np.argsort(similarities)[-top_k:][::-1]

        return self.df.iloc[top_indices]


###############################################################
# SIMPLE HR ASSISTANT (NO LLM)
###############################################################

class HRAssistant:

    def __init__(self):

        self.search_engine = SemanticSearchEngine(
            "hr_policies.csv"
        )


    def answer_query(self, query):

        results = self.search_engine.search(query)

        context = "\n".join(results["text"].tolist())

        return f"""
Relevant HR Policy Matches:

{context}
"""


###############################################################
# CLI LOOP
###############################################################

def main():

    assistant = HRAssistant()

    print("HR Assistant Ready")
    print("Type 'exit' to quit\n")

    while True:

        query = input("Ask HR question: ").strip()

        if query.lower() == "exit":
            break

        if not query:
            continue

        print("\nSearching policies...\n")

        answer = assistant.answer_query(query)

        print(answer)


if __name__ == "__main__":
    main()

Dataset missing. Creating sample dataset...
Sample HR dataset created.
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing embeddings...
Embeddings ready.

HR Assistant Ready
Type 'exit' to quit

